# check_equilibrium

Visual front-end for `thermo_analyzer.py`. Parses a LAMMPS NPT log, then plots:

1. **Per-point equilibration** — volume trace of a B block, the analysis window (last `EQ_WINDOW_FRAC` of B), the two-halves means, the drift fit, and the A→B continuity reference.
2. **Equilibration overview** — `drift_frac`, `half_frac`, `cont_frac`, `N_eff` vs density, colored by PASS/WARN.
3. **Fluctuation response functions** — `B_T`, `B_S`, `Cp`, `Cv`, `alpha_P`, `gamma`, sound speed vs density, with WARN points flagged.

All physics comes from `thermo_analyzer` directly (single source of truth).

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# thermo_analyzer.py lives next to this notebook.
sys.path.insert(0, os.path.abspath('.'))
import thermo_analyzer as ta

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## Parse a log

Set `LOG` to the log you want. System / atom count / steps-per-part are auto-detected.

In [ ]:
LOG = 'out-load-npt-96k-1a.o4495902'   # the long compression log

points, meta = ta.parse_log(LOG)
# Keep only density points whose B (production) block ran to completion.
pts = [p for p in points if p.b is not None and p.b.is_complete]
print(f"system={meta['system']}  n_atoms={meta['n_atoms']}  steps/part={meta['steps_per_part']}")
print(f"{len(points)} density points, {len(pts)} with a complete B block")
print(f"equilibration window = last {int(ta.EQ_WINDOW_FRAC*100)}% of each B block")

## 1. Per-point equilibration

Pick a density point and inspect its B block. The shaded region is the equilibration window.

In [ ]:
def plot_point_equilibration(pt):
    b = pt.b
    step, V = b.step, b.volume
    w0 = int(V.size * (1.0 - ta.EQ_WINDOW_FRAC))
    ws, win = step[w0:], V[w0:]
    r = ta.compute_equilibration(pt)

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))

    ax = axs[0]
    ax.plot(step, V, lw=0.6, color='0.75', label='B block')
    ax.axvspan(ws[0], ws[-1], color='C0', alpha=0.10, label='window')
    ax.plot(ws, win, lw=0.7, color='C0')
    h = win.size // 2
    ax.hlines(win[:h].mean(), ws[0], ws[h - 1], color='C1', lw=2.5, label='1st half mean')
    ax.hlines(win[h:].mean(), ws[h], ws[-1], color='C3', lw=2.5, label='2nd half mean')
    c = np.polyfit(ws.astype(float), win, 1)
    ax.plot(ws, np.polyval(c, ws.astype(float)), 'k--', lw=1.2, label='drift fit')
    if pt.a is not None:
        a = pt.a
        ah = a.volume.size // 2
        ax.plot(a.step, a.volume, lw=0.5, color='0.85')
        ax.hlines(a.volume[ah:].mean(), a.step[ah], a.step[-1], color='C2', lw=2, label='A tail mean')
    ax.set_xlabel('step'); ax.set_ylabel(r'Volume ($\AA^3$)')
    ax.set_title(f"rho={r['density']:.3f} g/cc   {r['status']}")
    ax.legend(fontsize=7)

    axs[1].plot(step, b.temperature, lw=0.5)
    axs[1].axvspan(ws[0], ws[-1], color='C0', alpha=0.10)
    axs[1].set_xlabel('step'); axs[1].set_ylabel('Temperature (K)')

    axs[2].plot(step, b.pressure, lw=0.5)
    axs[2].axvspan(ws[0], ws[-1], color='C0', alpha=0.10)
    axs[2].set_xlabel('step'); axs[2].set_ylabel('Pressure (GPa)')

    txt = (f"drift_frac={r['drift_frac']:.3f} (<{ta.EQ_DRIFT_FRAC_MAX})\n"
           f"half_frac={r['half_frac']:.3f} (<{ta.EQ_SHIFT_FRAC_MAX})\n"
           f"cont_frac={r['cont_frac']:.3f} (<{ta.EQ_SHIFT_FRAC_MAX})\n"
           f"tau_int={r['tau_int']:.1f}  N_eff={r['N_eff']:.0f} (>{ta.EQ_MIN_NEFF:.0f})")
    axs[0].text(0.02, 0.02, txt, transform=axs[0].transAxes, fontsize=7,
                va='bottom', ha='left', bbox=dict(boxstyle='round', fc='w', alpha=0.8))
    fig.tight_layout()
    return r

# A point that passes (low density) and one in the sluggish band.
_ = plot_point_equilibration(pts[0])
_ = plot_point_equilibration(pts[len(pts) // 2])

plt.show()

## 2. Equilibration overview vs density

Each metric vs density; dashed lines are the PASS thresholds. Red = WARN.

In [ ]:
eq = [ta.compute_equilibration(p) for p in pts]
rho = np.array([r['density'] for r in eq])
is_warn = np.array([r['status'] == 'WARN' for r in eq])
col = np.where(is_warn, 'C3', 'C2')

fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
panels = [
    ('drift_frac', ta.EQ_DRIFT_FRAC_MAX, 'drift_frac'),
    ('half_frac',  ta.EQ_SHIFT_FRAC_MAX, 'half_frac'),
    ('cont_frac',  ta.EQ_SHIFT_FRAC_MAX, 'cont_frac'),
    ('N_eff',      ta.EQ_MIN_NEFF,       'N_eff'),
]
for ax, (key, thr, label) in zip(axs.ravel(), panels):
    y = np.array([r[key] for r in eq])
    ax.scatter(rho, y, c=col, s=14)
    ax.axhline(thr, ls='--', color='k', lw=1)
    ax.set_ylabel(label)
    if key == 'N_eff':
        ax.set_yscale('log')
for ax in axs[1]:
    ax.set_xlabel('density (g/cc)')
n_warn = int(is_warn.sum())
fig.suptitle(f'Equilibration: {len(pts) - n_warn} PASS / {n_warn} WARN', y=1.01)
fig.tight_layout()

## 3. Fluctuation response functions vs density

Open red markers = points flagged WARN by the equilibration check (treat with care).

In [ ]:
fl = [ta.compute_fluctuations(p.b, meta['n_atoms']) for p in pts]
frho = np.array([r['density'] for r in fl])

def fl_arr(key):
    return np.array([r[key] for r in fl])

panels = [
    ('B_T', 'B_T (GPa)'), ('B_S', 'B_S (GPa)'),
    ('Cp_atom', 'Cp (k_B/atom)'), ('Cv_atom', 'Cv (k_B/atom)'),
    ('alpha_P', 'alpha_P (1/K)'), ('gamma', 'Gruneisen'),
    ('sound', 'sound (m/s)'), ('kappa_T', 'kappa_T (1/GPa)'),
]
fig, axs = plt.subplots(4, 2, figsize=(12, 13), sharex=True)
for ax, (key, label) in zip(axs.ravel(), panels):
    y = fl_arr(key)
    ax.plot(frho[~is_warn], y[~is_warn], 'o', ms=4, color='C2', label='PASS')
    ax.plot(frho[is_warn], y[is_warn], 'o', ms=4, mfc='none', color='C3', label='WARN')
    ax.set_ylabel(label)
axs[0, 0].legend(fontsize=8)
for ax in axs[-1]:
    ax.set_xlabel('density (g/cc)')
fig.suptitle(f'NPT fluctuation response functions — {meta["system"]} {meta["n_atoms"]} atoms', y=1.005)
fig.tight_layout()

## 4. (optional) Multi-sample average with error bars

If you ran `average_samples.py`, load `average/analysis/fluctuations` (means) + `.err` (SEM) and plot with error bars. Adjust `AVG_DIR`.

In [ ]:
AVG_DIR = '../average/analysis'
mean_path = os.path.join(AVG_DIR, 'fluctuations')
err_path = mean_path + '.err'

if os.path.isfile(mean_path) and os.path.isfile(err_path):
    cols = ta._FLUCT_COLUMNS
    m = np.loadtxt(mean_path)
    e = np.loadtxt(err_path)
    ci = {c: i for i, c in enumerate(cols)}
    x = m[:, ci['density']]
    fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    for ax, key in zip(axs.ravel(), ['B_T', 'Cp_atom', 'alpha_P', 'sound']):
        ax.errorbar(x, m[:, ci[key]], yerr=e[:, ci[key]], fmt='o', ms=3,
                    color='C0', ecolor='0.6', capsize=2)
        ax.set_ylabel(key)
    for ax in axs[1]:
        ax.set_xlabel('density (g/cc)')
    fig.suptitle('5-sample mean ± SEM', y=1.01)
    fig.tight_layout()
else:
    print(f'No averaged files at {AVG_DIR}; run average_samples.py first.')